# Automating Labeling

We use the open source framework called Snorkel to programmatically label a (raw) dataset to produce an adequate trained dataset.

This notebook is based on the [SnorkelSpam Tutorial](https://github.com/snorkel-team/snorkel-tutorials/blob/master/spam), which is described as: "We consider a canonical machine learning problem: classifying spam."


In [74]:
# Install packages
!python3 -m pip install pandas snorkel > /dev/null 2>&1
# Download the spacy English model
!python3 -m spacy download en_core_web_sm > /dev/null 2>&1

%matplotlib inline

import pandas as pd
from enum import Enum

## Load Sample Data

Let's load and label a YouTube comments dataset.

In [75]:
df_train = pd.read_csv('../../data/youtube_train.csv')
df_test = pd.read_csv('../../data/youtube_test.csv')

# Let's look at 15 random data points from the training dataset.

print(df_train[['author', 'text', 'video']].sample(15, random_state=42))

                 author                                               text  \
468      Brian Schultes                  this song never get's old &lt;3 ﻿   
332      OFFICIAL LEXIS  Hi everyone! Do you like music? Then why not c...   
946        Patrik Gybka        SUBSCRIBE MY CHANNEL PLEASE LOL PRO PLAYS)﻿   
380   William Fernandez  Katy Perry - Roar (Official): http://youtu.be/...   
99               eSkiip  This video will get to 2 billion just because ...   
1534   Daniel Tabatabai                           still listening in 2015﻿   
1118          Lolumad66  Best song ever made i swear :D i still hear ev...   
1030          Hoye Boyz  HAHAA THIS DANCE IS TIGHTTTT<br /><br />I know...   
939           alanluna3                  Check out this video on YouTube:﻿   
303      Young IncoVEVO  Check out my Music Videos! Fuego - U LA LA Rem...   
381       YULIOR ZAMORA  I    loved        it           so       much  ...   
1503             Sylith      Rihanna and Eminem together are uns

## Writing Labeling Functions

To create a snorkel labeling function, we can use the `@labeling_function` decorator.

We embed the labeling rules inside a `@labeling_function`. We start with a set of simple labeling rules. Let's say that for comments that has 'check out' in the text, it likely to be a spam. But we want to compare to matching the comment text to 'text' so that we can understand the tradeoff between having more coverage vs accuracy. So we can play around with the data with labeling functions by having 2 rules:

* The check rule that matches 'check'.
* The check_out rule that matches 'check out'.


In [76]:
from snorkel.labeling import labeling_function, PandasLFApplier

class Label(Enum):
  ABSTAIN = -1
  HAM = 0
  SPAM = 2


# Define a labeling function.

@labeling_function()
def filter_check(x):
    if 'check' in x['text'].lower():
        return Label.SPAM.value   # Must return an integer, hence .value
    
    return Label.ABSTAIN.value


@labeling_function()
def filter_check_out(x):
    if 'check out' in x['text'].lower():
        return Label.SPAM.value
    
    return Label.ABSTAIN.value 


# Chain the labeling functions together.
lfs = [filter_check, filter_check_out]

# Apply the lf's to the df.
applier = PandasLFApplier(lfs=lfs)
L_train = applier.apply(df=df_train) 
print(L_train)

100%|██████████| 1586/1586 [00:00<00:00, 159524.37it/s]

[[-1 -1]
 [-1 -1]
 [ 2 -1]
 ...
 [ 2  2]
 [ 2 -1]
 [ 2  2]]


### Use of apply

The `apply` method returns a label matrix, which is an array of arrays, which each item corresponds the result of the labeling function.

So if our training dataset `df_train` looks like this:

| id  | author         | text                                              |
|-----|----------------|---------------------------------------------------|
| 468 | Brian Schultes | his song never get's old &lt;3                    |
| 303 | Young IncoVEVO | Check out my Music Videos! Fuego - U LA LA Rem... |

Running the following:

```
applier = PandasLFApplier(lfs=lfs)
L_train = applier.apply(df=df_train) 
```

gives us the following output:

```
[
  [-1 -1]   # No 'check' or 'check out'
  [ 2  2]   # Found 'check' and 'check out'
]
```

## Stats of the Labeling Process

We can get statistics on the labeling process, like coverage ie. % of the data points that match a rule.

In [77]:
coverage_check, coverage_check_out = (L_train != Label.ABSTAIN.value).mean(axis=0)

print('% of trained dataset that contains')
print(f'check coverage: {coverage_check * 100:.1f}%')
print(f'check out coverage: {coverage_check_out * 100:.1f}%')


% of trained dataset that contains
check coverage: 25.8%
check out coverage: 21.4%


## Deeper Dive into Labeling

Because we are automating the labeling by mass based on the hypotheses (or our intuition) that comments containing 'check out' and 'check' usually means that the comment is a spam, we need to perform some analysis of the data to validate that our hypothesis and assumptions are valid enough. Snorkel provides a few ways to analyze the performance of the labeling processing so that may tune the labeling performance.

Here are the definitions of the stats in the `LDAnalysis` summary:

| Stats     | Description                                                                                                                               |
|-----------|-------------------------------------------------------------------------------------------------------------------------------------------|
| Polarity  | The set of unique labels this LF outputs (so far we have SPAM and ABSTAIN, but the system excludes ABSTAIN, hence we have 1 unique label) |
| Coverage  | The fraction of the dataset the LF labels                                                                                                 |
| Overlaps  | The fraction of the dataset where this LF and at least one other LF label                                                                 |
| Conflicts | The fraction of the dataset where this LF and at least one other LF label and disagree                                                    |

From the previous run, which is backed by `LFAnalysis`, we see that the 'check out' rule has 21.43% coverage while 'check' rule has 25.78% ie. more comments are labeled as spam by the 'check' rule than the other rule.


In [78]:
from snorkel.labeling import LFAnalysis

# We can perform a deeper analysis of the labeling coverage using LFAnalysis.
LFAnalysis(L=L_train, lfs=lfs).lf_summary()

,j,Polarity,Coverage,Overlaps,Conflicts
filter_check,0,[2],0.257881,0.214376,0.0
filter_check_out,1,[2],0.214376,0.214376,0.0


### Get Samples of the Labeled Dataset

Let's dive deeper by getting a sample of the training dataset that is labeled spam marked by the 'check' rule.

#### Examine the SPAM

In [79]:
# NOTE: 'check' rule corresponds to index 0, 'check out' rule to index 1. See the order of the lfs list.

df_train.iloc[L_train[:, 0] == Label.SPAM.value].sample(100, random_state=42)

,Unnamed: 0,author,date,text,label,video
941,241,Sophie Flores,2015-01-10T18:17:21.145000,Check out this video on YouTube:﻿,1,2
809,109,Helperitza,2015-02-04T10:23:50.394000,Check out this video on YouTube:﻿,1,2
761,61,ＯＧＶＡＤＥＲ,2014-09-29T02:54:58.986000,man check out the raps on my channel im better...,1,2
693,343,Nathan Waterhouse,2014-09-16T14:21:04,Please check out my acoustic cover channel :) ...,1,1
511,161,MarianMusicChannel,2014-08-24T03:57:52,"Hello! I'm Marian, I'm a singer from Venezuela...",1,1
...,...,...,...,...,...,...
951,251,Nana Diaz,2015-04-26T20:44:52.830000,Check out this video on YouTube:﻿,1,2
1537,399,Ked Woodly,NaN,COFFEE ! LOVERS ! PLEASE ! READ ! Check out a ...,1,3
772,72,((A.B)),2014-10-04T01:35:02.414000,LMFAO is CRAZY DOPE!!! CHeck out my music on m...,1,2
759,59,sahir omran,2015-01-27T09:43:48.741000,Check out this video on YouTube:الإعلانات<br /...,1,2


The sample so far looks valid, matching our intuition so far. Most of the positives look fine and no significant false positives.

#### Examine the SPAM AND ABSTAIN 

Let's dive deeper by examining where 'check_out' rule produces ABSTAIN and 'check' produces SPAM. 

In [80]:
from snorkel.analysis import get_label_buckets

buckets = get_label_buckets(L_train[:, 0], L_train[:, 1])
df_train.iloc[buckets[(Label.SPAM.value, Label.ABSTAIN.value)]].sample(10, random_state=42)

,Unnamed: 0,author,date,text,label,video
167,167,Brandon Pryor,2014-01-19T00:36:25,I dont even watch it anymore i just come here ...,0,0
2,2,Phuc Ly,2014-01-20T15:27:47,go here to check the views :3﻿,0,0
511,161,MarianMusicChannel,2014-08-24T03:57:52,"Hello! I'm Marian, I'm a singer from Venezuela...",1,1
38,38,BIGMOFO Tonkatruck,2014-11-12T06:26:42,just came to check the view count﻿,0,0
612,262,Sam Klein,2014-08-31T03:52:29,"She named the tiger Kitty Purry No, seriously...",0,1
146,146,Bob Kanowski,2013-11-28T12:33:27,i turned it on mute as soon is i came on i jus...,0,0
82,82,Freddie Barton,2014-11-06T15:31:35,CHECK MY CHANNEL FOR MY NEW SONG 'STATIC'!! YO...,1,0
261,261,Dana Matich,2014-11-08T03:32:55,Hey guys! Check this out: Kollektivet - Don't ...,1,0
429,79,SergiBeCa Official,2014-09-12T03:39:36,Can you check my videos please? Don't hate me ...,1,1
103,103,Jason Provencal,2014-01-19T16:15:54,Why dafuq is a Korean song so big in the USA. ...,0,0


From the result, we discover that phrases like 'check this out', 'check *** out', etc are mislabeled as ABSTAIN. So can we improve our coverage and accuracy?

(To be continued)